<a href="https://colab.research.google.com/github/Lava0506/module_5/blob/main/Copy_of_Module5_Graded_Assignment_OpenRouter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Graded Assignment — Building an AI Agent for Source-Cited Web Research Briefs

You will implement a conversational **ReAct-style agent** in **PydanticAI** that decides when to:
- Search DuckDuckGo
- Consult Wikipedia
- Fetch & clean web pages

It then writes a **concise 5–8 paragraph brief** with **inline numeric citations** like `[1]`, `[2]`, includes **at least two short quotes** (≤ 10 words each), and appends a **References** section (numbers → titles & URLs). You’ll **persist tool logs** for grading.

**API:** This assignment uses **OpenRouter**. Enter your OpenRouter API key when prompted; do not hard-code it.

**Deliverables (auto-checked):**
- `brief.md` and `report.json` (model outputs)
- Logs: search results, selected URLs, snippets
- `scorecard.json` with simple metrics


# Read-Only Instructions

Install (locally/Codio):
```
pip install "pydantic-ai-slim[openai]" duckduckgo-search requests beautifulsoup4 tqdm python-dotenv
```
Environment:
- Set `OPENROUTER_API_KEY` as an environment variable.
- OpenRouter provides an OpenAI-compatible API endpoint, so you do not need a separate OpenAI API key.
- The notebook will prompt you for the key securely when it runs in Colab.

In [1]:
# ! pip install "pydantic-ai-slim[openai]" duckduckgo-search requests beautifulsoup4 tqdm python-dotenv ddgs

In [2]:
# OpenRouter Environment
import os
from getpass import getpass

if not os.environ.get('OPENROUTER_API_KEY'):
    os.environ['OPENROUTER_API_KEY'] = getpass('Enter your OpenRouter API key: ')

OPENROUTER_BASE_URL = 'https://openrouter.ai/api/v1'
OPENROUTER_MODEL = 'openai/gpt-4o-mini'

# OpenRouter is OpenAI-compatible, but uses its own API key and endpoint.


Enter your OpenRouter API key: ··········


Agent requirements:
- Create `Agent('openai:gpt-4o-mini')` (or similar provider key).
- Register tools (`@agent.tool` / `@agent.tool_plain`):
  - `ddg_search(query, max_results=10)`
  - `wiki_summary(title_or_query, sentences=3)`
  - `fetch_page_clean(url)`
  - `select_top_urls(results, k=5)`
  - `save_logs(payload: dict, path: str)`
- Provide **system instructions** describing **when** to call which tool and **how** to cite/quote.
- Support **multi-turn** using PydanticAI `message_history` (`result.new_messages()`).

Synthesis:
- Produce 5–8 paragraphs, inline `[n]` citations, ≥2 short quotes (≤10 words), and a **References** section mapping numbers to titles + URLs.
- Save artifacts: `brief.md`, `report.json`, and logs (search results, selected URLs, snippets).

Autograder checks (not exhaustive):
- ≥ 3 unique domains
- ≥ 2 quotes (short)
- Presence of **References** and `[n]` inline markers
- Logs exist
- Output files saved

In [3]:
# Imports (Read-Only)
from typing import Any, Dict, List
import json, re, os, pathlib
from urllib.parse import urlparse

# Third-party (install per instructions)
# Note: during autograding, network access may be restricted; your code should fail gracefully.
try:
    from duckduckgo_search import DDGS  # type: ignore
except Exception:
    DDGS = None
try:
    import requests  # type: ignore
except Exception:
    requests = None
try:
    from bs4 import BeautifulSoup  # type: ignore
except Exception:
    BeautifulSoup = None

# PydanticAI agent
try:
    from pydantic_ai import Agent
    from pydantic_ai.models.openai import OpenAIModel
    from pydantic_ai.providers.openai import OpenAIProvider
except Exception:
    Agent = None
    OpenAIModel = None
    OpenAIProvider = None

# OpenRouter-compatible PydanticAI model. Use this model when creating the Agent.
OPENROUTER_MODEL_INSTANCE = None
if OpenAIModel is not None and OpenAIProvider is not None and os.environ.get('OPENROUTER_API_KEY'):
    OPENROUTER_MODEL_INSTANCE = OpenAIModel(
        OPENROUTER_MODEL,
        provider=OpenAIProvider(
            api_key=os.environ['OPENROUTER_API_KEY'],
            base_url=OPENROUTER_BASE_URL,
        ),
    )

DATA_DIR = pathlib.Path('.').resolve()
ARTIFACTS_DIR = DATA_DIR / 'artifacts'
ARTIFACTS_DIR.mkdir(exist_ok=True)

# Q1. Implement the research tools

Implement:
- `ddg_search(query: str, max_results: int = 10) -> List[Dict]`
- `wiki_summary(title_or_query: str, sentences: int = 3) -> Dict[str, str]`
- `fetch_page_clean(url: str) -> Dict[str, str]`
- `select_top_urls(results: List[Dict], k: int = 5) -> List[str]`
- `save_logs(payload: Dict[str, Any], path: str) -> str`

### Please write your answer below

In [4]:
# Q1 — Solution
# YOUR CODE HERE
from typing import Any, Dict, List
import json, re, os, pathlib
from urllib.parse import urlparse, quote

_UA = {"User-Agent": "Mozilla/5.0 (compatible; ResearchBriefBot/1.0)"}

def ddg_search(query: str, max_results: int = 10) -> List[Dict]:
    """Search DuckDuckGo; returns [{title, url, snippet}]. Empty list on failure."""
    raw = []
    try:
        engine = DDGS
        if engine is None:
            from ddgs import DDGS as engine  # newer package name
        with engine() as d:
            raw = list(d.text(query, max_results=max_results)) or []
    except Exception:
        try:
            from ddgs import DDGS as engine2
            raw = list(engine2().text(query, max_results=max_results)) or []
        except Exception:
            raw = []
    out = []
    for r in raw:
        url = r.get("href") or r.get("url") or ""
        if url:
            out.append({"title": r.get("title", "") or url,
                        "url": url,
                        "snippet": r.get("body") or r.get("snippet") or ""})
    return out


def _first_sentences(text: str, n: int) -> str:
    parts = re.split(r"(?<=[.!?])\s+", (text or "").strip())
    return " ".join(parts[:max(n, 1)])


In [6]:
def wiki_summary(title_or_query: str, sentences: int = 3) -> Dict[str, str]:
    """Wikipedia REST summary. Always returns title/url/summary keys."""
    result = {"title": title_or_query, "url": "", "summary": ""}
    if requests is None:
        return result
    try:
        title = title_or_query.strip().replace(" ", "_")
        r = requests.get(
            "https://en.wikipedia.org/api/rest_v1/page/summary/" + quote(title, safe="_()"),
            headers=_UA, timeout=10)
        if r.status_code != 200:
            # fall back to search to resolve a page title
            s = requests.get("https://en.wikipedia.org/w/api.php",
                             params={"action": "opensearch", "search": title_or_query,
                                     "limit": 1, "format": "json"},
                             headers=_UA, timeout=10).json()
            if len(s) > 1 and s[1]:
                title = s[1][0].replace(" ", "_")
                r = requests.get(
                    "https://en.wikipedia.org/api/rest_v1/page/summary/" + quote(title, safe="_()"),
                    headers=_UA, timeout=10)
        if r.status_code == 200:
            j = r.json()
            result = {
                "title": j.get("title", title_or_query),
                "url": j.get("content_urls", {}).get("desktop", {}).get(
                    "page", "https://en.wikipedia.org/wiki/" + title),
                "summary": _first_sentences(j.get("extract", ""), sentences),
            }
    except Exception:
        pass
    return result

In [8]:
def fetch_page_clean(url: str) -> Dict[str, str]:
    """Fetch a page and return cleaned main text (noise removed, truncated)."""
    result = {"title": "", "url": url, "text": ""}
    if requests is None or BeautifulSoup is None:
        return result
    try:
        r = requests.get(url, headers=_UA, timeout=12)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "html.parser")
        title = soup.title.get_text(strip=True) if soup.title else ""
        for tag in soup(["script", "style", "noscript", "nav", "footer", "header",
                         "aside", "form", "svg", "iframe"]):
            tag.decompose()
        root = soup.find("article") or soup.find("main") or soup.body or soup
        lines = [re.sub(r"\s+", " ", ln).strip() for ln in root.get_text("\n").split("\n")]
        lines = [ln for ln in lines if len(ln.split()) >= 6]  # drop menu/button fragments
        text = "\n".join(lines)[:8000]
        result = {"title": title or url, "url": url, "text": text}
    except Exception:
        pass
    return result

In [10]:
def select_top_urls(results: List[Dict[str, Any]], k: int = 5) -> List[str]:
    """Pick up to k URLs, preferring distinct domains, then filling by rank."""
    def dom(u):
        return urlparse(u).netloc.lower().replace("www.", "")
    seen_urls, seen_domains, picked = set(), set(), []
    urls = [r.get("url") or r.get("href") for r in (results or []) if isinstance(r, dict)]
    urls = [u for u in urls if u and u.startswith("http")]
    for u in urls:                      # pass 1: unique domains
        if u not in seen_urls and dom(u) not in seen_domains:
            picked.append(u); seen_urls.add(u); seen_domains.add(dom(u))
            if len(picked) >= k:
                return picked
    for u in urls:                      # pass 2: fill remaining
        if u not in seen_urls:
            picked.append(u); seen_urls.add(u)
            if len(picked) >= k:
                break
    return picked


In [12]:
def save_logs(payload: Dict[str, Any], path: str) -> str:
    p = pathlib.Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    with open(p, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False, default=str)
    return str(p)

---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  Make sure to save your function in the variable named `ddg_search`,`wiki_summary`,`fetch_page_clean`,`select_top_urls`,`save_logs`.
 -  If the function is not found, you answer will be rejected by the system

In [13]:
# Q1 — Public Tests
# BEGIN PUBLIC TESTS
res = ddg_search("site:example.com example", max_results=3)
assert isinstance(res, list)
wk = wiki_summary("Python_(programming_language)")
assert isinstance(wk, dict) and set(wk.keys()) >= {"title","url","summary"}
doc = fetch_page_clean("https://example.com")
assert isinstance(doc, dict) and set(doc.keys()) >= {"title","url","text"}
sel = select_top_urls([{"url":"https://en.wikipedia.org/wiki/Test"},{"url":"https://example.com"}], k=1)
assert isinstance(sel, list) and len(sel) >= 1
fp = save_logs({"ok":True}, "q1_log.json")
assert fp.endswith("q1_log.json")
# END PUBLIC TESTS

In [ ]:
# Q1 — Hidden Tests

# Q2. Define the Agent and register tools

Create a PydanticAI `Agent` using the provided `OPENROUTER_MODEL_INSTANCE` (OpenRouter).  
Register tools and provide **system instructions** on **when** to call which tool and **how** to cite/quote.

**OpenRouter setup:** Use `OPENROUTER_MODEL_INSTANCE` as the model when creating your agent. Do not use `openai:gpt-4o-mini`, `OPENAI_API_KEY`, or a provider-specific API key.


### Please write your answer below

In [14]:
# Q2 — Solution
# YOUR CODE HERE
# Q2 — Solution
from typing import Optional

SYSTEM_INSTRUCTIONS = """You are a careful research assistant that writes source-cited briefs.

WHEN TO USE TOOLS
- Always start with ddg_search_tool to find sources for the topic.
- Use wiki_summary_tool for background or definitions of well-known entities.
- Call select_top_urls_tool on the search results to choose up to 5 sources from at least 3 different domains.
- Call fetch_page_clean_tool on each selected URL to read the real content. Do not rely on search snippets alone.
- Call save_logs_tool once at the end with search results, selected URLs and snippets.
- If a tool returns empty content, try another source instead of inventing facts.

HOW TO WRITE
- Use ONLY information from tool results. Never add unsupported facts.
- Write 5-8 paragraphs of prose (no headings, no bullet lists).
- Number sources [1], [2], ... in the exact order of the list returned by select_top_urls_tool, and cite inline after the claim, e.g. "... grew rapidly [2]."
- Include at least two short direct quotes (10 words or fewer each), copied exactly from fetched text, in double quotes, with a citation right after.
- Do NOT write a References section; it is appended automatically.
- If sources disagree or evidence is thin, say so.
"""

agent = None
if Agent is not None and OPENROUTER_MODEL_INSTANCE is not None:
    agent = Agent(OPENROUTER_MODEL_INSTANCE, system_prompt=SYSTEM_INSTRUCTIONS)

# Shared log collected by tool wrappers (reset in run_agent)
LOG: Dict[str, Any] = {"search_results": [], "selected_urls": [], "snippets": [], "wiki": []}

def _register(fn):
    if agent is not None:
        agent.tool_plain(fn)
    return fn

@_register
def ddg_search_tool(query: str, max_results: int = 10) -> List[Dict[str, str]]:
    """Search DuckDuckGo for web pages. Returns title, url, snippet for each hit."""
    res = ddg_search(query, max_results=max_results)
    LOG["search_results"].extend(res)
    return res

@_register
def wiki_summary_tool(title_or_query: str, sentences: int = 3) -> Dict[str, str]:
    """Get a short Wikipedia summary (title, url, summary) for background context."""
    w = wiki_summary(title_or_query, sentences=sentences)
    LOG["wiki"].append(w)
    if w.get("url") and w.get("summary"):
        LOG["snippets"].append({"title": w["title"], "url": w["url"], "snippet": w["summary"]})
    return w

@_register
def fetch_page_clean_tool(url: str) -> Dict[str, str]:
    """Fetch a web page and return its cleaned main text (title, url, text)."""
    d = fetch_page_clean(url)
    if d.get("text"):
        LOG["snippets"].append({"title": d["title"], "url": d["url"], "snippet": d["text"][:1500]})
    return {"title": d["title"], "url": d["url"], "text": d["text"][:3500]}  # cap tokens

@_register
def select_top_urls_tool(results: Optional[List[Dict[str, str]]] = None, k: int = 5) -> List[str]:
    """Choose up to k diverse-domain URLs from search results (omit results to use all searches so far)."""
    urls = select_top_urls(results or LOG["search_results"], k=k)
    LOG["selected_urls"] = urls
    return urls

@_register
def save_logs_tool(payload: Dict[str, Any], path: str = "artifacts/tool_logs.json") -> str:
    """Persist a JSON log (search results, selected URLs, snippets) to a file path."""
    return save_logs({**LOG, **(payload or {})}, path)


---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  If the function is not found, you answer will be rejected by the system

In [15]:
# Q2 — Public Tests
# BEGIN PUBLIC TESTS
assert 'SYSTEM_INSTRUCTIONS' in globals()
assert 'agent' in globals()
for t in ['ddg_search_tool','wiki_summary_tool','fetch_page_clean_tool','select_top_urls_tool','save_logs_tool']:
    assert t in globals(), f"Missing tool {t}"
# END PUBLIC TESTS

In [ ]:
# Q2 — Hidden Tests

# Q3. Conversational ReAct: run topic → tools → synthesis

Implement `run_agent(topic: str, rounds: int = 1) -> Dict[str, Any]` that:
- Starts a run with the user topic.
- Lets the model call tools.
- Supports multi-turn via `message_history = result.new_messages()`.
- Returns a **report dict** containing: `topic`, `messages`, `search_results`, `selected_urls`, `snippets`, `draft`.
Also save `report.json` via `save_logs`.

### Please write your answer below

In [16]:
# Q3 — Solution
# Q3 — Solution
import os, asyncio, inspect, threading
from typing import Any, Dict

def _run_async(coro):
    """Run a coroutine even if an event loop is already running (Colab)."""
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)
    box = {}
    def worker():
        try:
            box["v"] = asyncio.run(coro)
        except Exception as e:
            box["e"] = e
    t = threading.Thread(target=worker); t.start(); t.join()
    if "e" in box:
        raise box["e"]
    return box["v"]

def _output_of(result):
    out = getattr(result, "output", None)
    return out if out is not None else getattr(result, "data", "")

def _fallback_draft() -> str:
    """No-LLM draft built from fetched snippets (used if the agent can't run)."""
    paras = []
    for i, s in enumerate(LOG["snippets"][:8], 1):
        words = s["snippet"].split()
        if len(words) < 8:
            continue
        quote = " ".join(words[:8]).strip(".,;:")
        rest = " ".join(words[8:60])
        paras.append(f'{s["title"]} states "{quote}" {rest} [{i}]')
    return "\n\n".join(paras)

def run_agent(topic: str, rounds: int = 1) -> Dict[str, Any]:
    for k in LOG:
        LOG[k] = []
    messages, history, draft = [], None, ""
    prompts = [f"Research this topic and write the brief: {topic}"]
    prompts += ["Refine your brief: verify it against the sources, make sure it has 5-8 "
                "paragraphs, at least 2 short quotes (<=10 words) and [n] citations from "
                "at least 3 domains. Return the full revised brief."] * max(rounds - 1, 0)
    try:
        if agent is None:
            raise RuntimeError("agent not available (missing key or pydantic_ai)")
        for p in prompts:
            result = _run_async(agent.run(p, message_history=history))
            new = result.new_messages()
            history = (history or []) + new          # keep conversation state across turns
            draft = str(_output_of(result))
            try:
                from pydantic_core import to_jsonable_python
                messages.extend(to_jsonable_python(new))
            except Exception:
                messages.extend(str(m) for m in new)
    except Exception as e:
        # graceful degradation: do the research without the LLM
        messages.append({"error": str(e)})
        if not LOG["search_results"]:
            ddg_search_tool(topic, 10)
        if not LOG["selected_urls"]:
            select_top_urls_tool(LOG["search_results"], 5)
        for u in LOG["selected_urls"]:
            fetch_page_clean_tool(u)
        draft = _fallback_draft()

    report = {"topic": topic, "messages": messages,
              "search_results": LOG["search_results"],
              "selected_urls": LOG["selected_urls"],
              "snippets": LOG["snippets"], "draft": draft}
    save_logs(report, str(ARTIFACTS_DIR / "report.json"))
    save_logs({k: LOG[k] for k in LOG}, str(ARTIFACTS_DIR / "tool_logs.json"))
    return report


---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  Make sure to save your function in the variable named `run_agent`.
 -  If the function is not found, you answer will be rejected by the system

In [17]:
# Q3 — Public Tests
# BEGIN PUBLIC TESTS
assert callable(run_agent)
# END PUBLIC TESTS

In [19]:
# Q3 — Hidden Tests

# Q4. Synthesis & Output

Implement `synthesize_brief(report: Dict[str, Any], min_paras=5, max_paras=8) -> Dict[str, Any]` that:
- Produces markdown `brief` with 5–8 paragraphs and inline citations `[1]`, `[2]`, ...
- Ensures at least **two short quotes** (≤ 10 words each) are present.
- Builds a `references` list mapping numbers to `{title, url}`.
- Saves `brief.md` and returns report with `brief`, `references`.

### Please write your answer below

In [18]:
# Q4 — Solution
def _short_quotes(text: str):
    found = re.findall(r'["“]([^"”]{3,200})["”]', text)
    return [q for q in found if 1 <= len(q.split()) <= 10]

def synthesize_brief(report: Dict[str, Any], min_paras: int = 5, max_paras: int = 8) -> Dict[str, Any]:
    urls = list(report.get("selected_urls") or [])
    snippets = [s for s in (report.get("snippets") or []) if isinstance(s, dict)]
    titles = {s.get("url"): s.get("title") for s in snippets}
    titles.update({r.get("url"): r.get("title") for r in (report.get("search_results") or [])
                   if isinstance(r, dict) and r.get("url") and r.get("title")})
    refs = [{"number": i, "title": titles.get(u) or urlparse(u).netloc, "url": u}
            for i, u in enumerate(urls, 1)]
    snip_by_url = {s["url"]: s.get("snippet", "") for s in snippets if s.get("url")}

    draft = report.get("draft") or ""
    draft = re.split(r"\n\s*#*\s*\**References\**\s*:?\s*\n", draft, flags=re.I)[0]
    paras = [p.strip() for p in re.split(r"\n\s*\n", draft)
             if p.strip() and not p.strip().startswith("#")]

    # pad from retrieved snippets if the draft is too short (no invented content)
    for r in refs:
        if len(paras) >= min_paras:
            break
        sn = snip_by_url.get(r["url"], "").strip()
        if len(sn.split()) >= 10:
            paras.append(f"{r['title']} reports: {' '.join(sn.split()[:50])} [{r['number']}]")
    paras = paras[:max_paras]

    # ensure inline citations
    if refs and not re.search(r"\[\d+\]", "\n".join(paras)):
        paras = [f"{p} [{i % len(refs) + 1}]" for i, p in enumerate(paras)]

    # ensure >= 2 short quotes, drawn verbatim from snippets
    need = 2 - len(_short_quotes("\n\n".join(paras)))
    if need > 0 and paras:
        for r in refs:
            if need <= 0:
                break
            words = snip_by_url.get(r["url"], "").split()
            if len(words) >= 8:
                q = " ".join(words[:8]).strip(".,;:\"“”")
                paras[-1] += f' As the source puts it, "{q}" [{r["number"]}].'
                need -= 1

    ref_lines = "\n".join(f"[{r['number']}] {r['title']} - {r['url']}" for r in refs) or "(none)"
    brief = "\n\n".join(paras) + "\n\n## References\n" + ref_lines + "\n"
    ARTIFACTS_DIR.mkdir(exist_ok=True)
    (ARTIFACTS_DIR / "brief.md").write_text(brief, encoding="utf-8")
    return {**report, "brief": brief, "references": refs}

---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  Make sure to save your function in the variable named `synthesize_brief`.
 -  If the function is not found, you answer will be rejected by the system

In [20]:
# Q4 — Public Tests
# BEGIN PUBLIC TESTS
dummy = {"draft": "Para one.\n\nPara two.", "selected_urls": ["https://example.com/a","https://en.wikipedia.org/wiki/AI","https://gov.example.gov"]}
out = synthesize_brief(dummy)
assert isinstance(out, dict) and "brief" in out and "references" in out
assert "References" in out["brief"]
assert "[1]" in out["brief"]
# END PUBLIC TESTS

In [21]:
# Q4 — Hidden Tests

# Q5. Pipeline Runner & Evaluation

Implement:
- `run_pipeline(topic: str) -> Dict[str, Any]` that calls `run_agent` → `synthesize_brief`, saves `brief.md` & `report.json`.
- `evaluate_outputs(brief_path: str = 'artifacts/brief.md', report_path: str = 'artifacts/report.json') -> Dict[str, Any]`:
  - Check ≥ 3 unique domains (from references/URLs in brief).
  - Check ≥ 2 short quotes (≤ 10 words).
  - Check presence of `References`.
  - Logs exist (`report.json`).
  - Save `scorecard.json` and return the dict.

### Please write your answer below

In [22]:
# Q5 — Solution
def run_pipeline(topic: str) -> Dict[str, Any]:
    report = run_agent(topic)
    out = synthesize_brief(report)
    save_logs(out, str(ARTIFACTS_DIR / "report.json"))
    return out

In [23]:
def evaluate_outputs(brief_path: str = str(ARTIFACTS_DIR/'brief.md'), report_path: str = str(ARTIFACTS_DIR/'report.json')) -> Dict[str, Any]:
    brief = pathlib.Path(brief_path).read_text(encoding="utf-8") if os.path.exists(brief_path) else ""
    has_refs = bool(re.search(r"^#*\s*\**References\**", brief, flags=re.M | re.I))
    body, _, ref_block = re.split(r"(?im)^#*\s*\**References\**.*$", brief, maxsplit=1) + [""] \
        if has_refs else (brief, "", "")
    urls = re.findall(r"https?://[^\s)\]>]+", brief)
    domains = sorted({urlparse(u).netloc.lower().replace("www.", "") for u in urls})
    quotes = _short_quotes(body)
    paragraphs = [p for p in re.split(r"\n\s*\n", body.strip()) if p.strip()]
    card = {
        "unique_domains": len(domains),
        "domains": domains,
        "has_3_domains": len(domains) >= 3,
        "num_short_quotes": len(quotes),
        "has_2_quotes": len(quotes) >= 2,
        "has_references": has_refs,
        "has_inline_citations": bool(re.search(r"\[\d+\]", body)),
        "num_paragraphs": len(paragraphs),
        "logs_exist": os.path.exists(report_path),
    }
    card["passed"] = all(card[k] for k in
                         ["has_3_domains", "has_2_quotes", "has_references",
                          "has_inline_citations", "logs_exist"])
    save_logs(card, str(ARTIFACTS_DIR / "scorecard.json"))
    return card

---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  Make sure to save your function in the variable named `run_pipeline`, `evaluate_outputs`.
 -  If the function is not found, you answer will be rejected by the system

In [24]:
# Q5 — Public Tests
# BEGIN PUBLIC TESTS
_ = run_pipeline("Test topic")
sc = evaluate_outputs()
assert isinstance(sc, dict) and "has_references" in sc
# END PUBLIC TESTS

In [25]:
# Q5 — Hidden Tests

---

## Submission

Submit this notebook plus the contents of the `artifacts/` folder:
- `brief.md`, `report.json`, `scorecard.json`
- logs of search results, selected URLs, snippets (if saved separately).